### Blibiotecas e Funções

In [60]:
from config import (
    RAW_PATH,
    PROCESSED_PATH,
    ANALYTICS_PATH,
    FILES,
    TRANSFORM_CONFIG
)
from extractor import extract_all_files, inspect, check_duplicates, check_date_range
from transform import cast_types, fill_nulls, add_columns, filter_rows, joins
from storage import save_csv, save_parquet, load_layer, load_summary
from pathlib import Path

### EXTRACTOR

In [ ]:
print("\n===EXTRACT===")
dfs = extract_all_files(str(RAW_PATH), FILES)

check_duplicates(dfs["deliveries"], "delivery_id", "deliveries")
check_date_range(dfs["deliveries"], "shipped_date", "deliveries")


=== EXTRACT ===
[extract] deliveries.csv | 102000 linhas | 11 colunas
[extract] drivers.csv | 255 linhas | 3 colunas
[extract] hubs.csv | 10 linhas | 3 colunas
[validate] 'deliveries' - duplicatas em 'delivery_id': 2000
[validate] 'deliveries' — shipped_date: 2026-01-01 → 2026-05-01


### TRANSFORM

In [62]:
df = dfs["deliveries"].copy()
df["status"].unique()

<ArrowStringArray>
['canceled', 'delayed', 'delivered']
Length: 3, dtype: str

In [ ]:
print("\n===TRANSFORM===")
df = dfs["deliveries"].copy()

df = cast_types(df,    TRANSFORM_CONFIG["types"])
df = filter_rows(df,   TRANSFORM_CONFIG["filters"], "deliveries")
df = fill_nulls(df,    TRANSFORM_CONFIG["fill_nulls"])
df = add_columns(df,   TRANSFORM_CONFIG["derived_cols"])

df_drivers         = dfs["drivers"].copy()
df_drivers["name"] = df_drivers["name"].str.strip().str.title()
df_hubs            = dfs["hubs"].copy()
df_hubs["city"]    = df_hubs["city"].str.strip().str.title()
df_hubs["state"]   = df_hubs["state"].str.strip().str.upper()

df = joins(df, df_drivers, on="driver_id", cols=["driver_id", "name"])
df = joins(df, df_hubs,    on="hub_id",    cols=["hub_id", "city", "state"])
df = df.rename(columns={"name": "driver_name"})


=== TRANSFORM ===
[transform] 'shipped_date' -> datetime


[transform] 'delivered_date' -> datetime
[transform] 'distance_km' -> float
[transform] 'cost' -> float
[transform] 'customer_rating' -> float
[transform] 'deliveries' — filter {'status': ['delivered', 'in_transit', 'pending']}: 68049 linhas removidas
[transform] 'customer_rating' - 1732 nulos preenchidos com '0.0'
[transform] Coluna 'delivery_days' criada
[transform] Coluna 'cost_per_km' criada
[transform] enrich on 'driver_id' (left) - shape: (33951, 14)
[transform] enrich on 'hub_id' (left) - shape: (33951, 16)


### LOAD

In [ ]:
print("\n===LOAD===")
for name, df_raw in dfs.items():
    load_layer(df_raw, str(PROCESSED_PATH.parent), "processed/raw",       f"raw_{name}")

load_layer(df,         str(PROCESSED_PATH.parent), "processed/processed", "deliveries_transformed")


=== LOAD ===
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed/raw\parquet\raw_deliveries.parquet' — 102000 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed/raw\csv\raw_deliveries.csv' — 102000 linhas
[load] 'raw_deliveries' → camada 'processed/raw'
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed/raw\parquet\raw_drivers.parquet' — 255 linhas
[storage] CSV salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed/raw\csv\raw_drivers.csv' — 255 linhas
[load] 'raw_drivers' → camada 'processed/raw'
[storage] Parquet salvo em 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed/raw\parquet\raw_hubs.parquet' — 10 linhas
[storage] CSV salvo em 'C:\Users

### RESUMO

In [ ]:
print("\n=== RESUMO ===")
df_summary = load_summary({
    "raw": [
        str(PROCESSED_PATH / "raw" / "parquet" / "raw_deliveries.parquet"),
        str(PROCESSED_PATH / "raw" / "parquet" / "raw_drivers.parquet"),
        str(PROCESSED_PATH / "raw" / "parquet" / "raw_hubs.parquet"),
    ],
    "processed": [
        str(PROCESSED_PATH / "processed" / "parquet" / "deliveries_transformed.parquet"),
    ],
})
print(df_summary.to_string(index=False))


=== RESUMO ===
[storage] Parquet carregado de 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\raw\parquet\raw_deliveries.parquet' — 102000 linhas
[storage] Parquet carregado de 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\raw\parquet\raw_drivers.parquet' — 255 linhas
[storage] Parquet carregado de 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\raw\parquet\raw_hubs.parquet' — 10 linhas
[storage] Parquet carregado de 'C:\Users\Ceifas\OneDrive\Desktop\data-engineering-portfolio\projects\month02_Pipeline\Data\processed\processed\parquet\deliveries_transformed.parquet' — 33951 linhas
   camada                        arquivo  linhas  colunas    tamanho
      raw         raw_deliveries.parquet  102000       11 2232.69 KB
      raw            raw_drivers.parquet     255        3    5.16 KB
      raw               raw_hubs.parquet